# U3.L2 — Diffusion as a Special Case

**Session U3.L2** (lab, joint). Companion to the U3.L2 lecture notes (<https://adricortes.github.io/flow-matching-lectures/U3/U3-L2-notes.html>), which carry the
reasoning; this notebook is the part you execute.

You turn the CFM code of U3.L1 into a DDPM-style noise-prediction trainer, load one class-conditional MNIST
model, and sample from it twice: with a deterministic ODE and with the stochastic ancestral chain of DDPM.
Then you check your conversions against an independent library (Azula) and score the samples with FID.

### How to read this notebook

Clone the public repository (<https://github.com/adricortes/flow-matching-labs>) and open this notebook from
`labs/`. MNIST downloads itself into `data/` at the first run. This lab needs one package more than U3.L1,
Azula, installed without its declared dependencies but with the two its import needs:
`pip install --no-deps azula==0.11.1 einops==0.8.2 tqdm==4.67.1`. Without them, the first code cell stops, names the missing module, and prints that command.

A GPU makes Parts 2, 3 and 5 fast; Part 5 still samples for several minutes. On a CPU every part still runs: the live training is shorter, and Part 5
reads the reference FID table instead of scoring 10 000 samples, and says so.

Cells marked **TODO** are yours. Everything else is given code: run it, read it, go on. Prompt cells ask for two
to four sentences in your own words. Checkpoints pass or say what is wrong.

## Part 0 — Prelude: the promise, and the given code

> The diff stays small, but every changed line has a mathematical label: path, prediction coordinate, or time
> weighting. Keep the diff — it is the course thesis with its fine print visible.

| File | What it is |
|---|---|
| `common/unet.py` | the course U-Net, `UNet(...).forward(x, t, y=None)`, with $t$ a `(B,)` float tensor in $[0,1]$ |
| `common/eval.py` | the evaluation harness: `ref_stats`, `fid`, `panel`, and the pinned MNIST feature extractor |
| `common/odesolve.py` | the course's ODE solvers (`odeint`, `SolverOutput` with `.nfe`) |
| `common/payoff.py` | the plot conventions of U3.L1 |
| `common/weights/u3l2-cond-mnist.*` | the course checkpoint of today and its provenance record |
| `labs/u3l2/reference/` | its sanitized MLflow export and the reference FID table |

Run the next two cells. The second one prints the contract of the U-Net and shows the assert that keeps the
diffusion time arrow out of the network: the network accepts $t \in [0,1]$ with $t=0$ noise, and nothing else.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import copy
import difflib
import hashlib
import inspect
import json
import math
import re
import time
import warnings

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets

from common import eval as harness, odesolve, payoff
from common.unet import UNet

try:                        # Part 4 needs it: a missing install stops here, not in Part 4
    import azula.denoise
    import azula.noise
    import azula.sample
except ImportError as err:
    raise ImportError(f"The module {err.name!r} is missing (Azula needs it). Install with:  "
                      "pip install --no-deps azula==0.11.1 einops==0.8.2 tqdm==4.67.1") from err

warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_GPU = DEVICE.type == "cuda"
COURSE_DATA = ROOT / "scripts" / "python" / "data"
DATA_DIR = COURSE_DATA if (COURSE_DATA / "MNIST").exists() else ROOT / "data"
CKPT = ROOT / "common" / "weights" / "u3l2-cond-mnist"
REF = ROOT / "labs" / "u3l2" / "reference"
torch.manual_seed(0); np.random.seed(0)
print("torch", torch.__version__, "| device", DEVICE, "| MNIST in", DATA_DIR)

In [ ]:
print(UNet.forward.__doc__)
probe = UNet(in_ch=1, base_ch=16, ch_mults=(1, 2, 2), attn_resolutions=(7,), num_classes=11)
try:
    probe(torch.zeros(2, 1, 28, 28), torch.tensor([0.5, 1.2]), torch.tensor([3, 10]))
except ValueError as err:
    print("ValueError:", err)
print()
print(harness.fid.__doc__.split("Implements")[0].strip())

In [ ]:
K = 1000            # grid size: the DDPM chain has K steps
BETA_MIN = 0.0001     # the first beta of Ho et al. (2020)
BETA_MAX = 0.02     # the last beta of Ho et al. (2020)
NULL_LABEL = 10     # the extra row of the label table (used by cfg_dropout)
N_TRAIN = 50000    # the rest of the MNIST training split is the validation set


def load_mnist(split: str) -> tuple[torch.Tensor, torch.Tensor]:
    """MNIST as (N, 1, 28, 28) float32 in [-1, 1] and (N,) long labels.

    split: "train" (50 000 rows of a seeded permutation of the training split), "val" (the other
    10 000; used for every choice of this build), or "test" (read once, by the reported FID).
    """
    ds = datasets.MNIST(str(DATA_DIR), train=(split != "test"), download=not (DATA_DIR / "MNIST").exists())
    x = ds.data.float().div(255.0).unsqueeze(1) * 2.0 - 1.0
    y = ds.targets.long()
    if split == "test":
        return x, y
    perm = torch.randperm(x.shape[0], generator=torch.Generator().manual_seed(0))
    rows = perm[:N_TRAIN] if split == "train" else perm[N_TRAIN:]
    return x[rows], y[rows]


def make_unet(cfg: dict) -> UNet:
    """The given `unet-skeleton` at the lab's configuration, with one label row per digit plus the null label."""
    m = cfg["model"]
    return UNet(in_ch=1, base_ch=m["base_ch"], ch_mults=tuple(m["ch_mults"]),
                attn_resolutions=tuple(m["attn_resolutions"]), num_classes=m["num_classes"] + 1)


def drop_labels(y: torch.Tensor, p: float, gen: torch.Generator) -> torch.Tensor:
    """Replace each label by NULL_LABEL with probability p (the `cfg_dropout` field)."""
    mask = torch.rand(y.shape, generator=gen) < p
    return torch.where(mask, torch.full_like(y, NULL_LABEL), y)


def move_head(net: nn.Module, scale: float = 0.1, seed: int = 0) -> nn.Module:
    """A copy of the network whose zero-initialized head is moved by a fixed, stated amount.

    With conv_out at exactly zero, every gradient upstream of it is exactly zero, and a check
    there measures nothing (MEMORY, U0.L4). The ritual therefore moves the head first.
    """
    out = copy.deepcopy(net)
    gen = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        for p in out.conv_out.parameters():
            p.add_(scale * torch.randn(p.shape, generator=gen, dtype=p.dtype))
    return out

## Part 1 — Path, coordinate, and weighting

Three things change against your `cfm-scratch` code of U3.L1, and nothing else:

1. **path**: the CondOT line $X_t = tX_1 + (1-t)X_0$ becomes a fixed grid of $K$ VP points
   $X_{t_k} = \alpha_k X_1 + \sigma_k X_0$;
2. **prediction coordinate**: the network predicts the noise sample $X_0$ instead of the velocity $X_1 - X_0$;
3. **time weighting**: either the weight that velocity CFM induces on the noise residual, or no weight at all.

### TODO 1.1 — the grid, on the FM arrow

DDPM numbers its chain $k = 0$ (data) to $k = K$ (noisiest). The class below translates that numbering **once**:
row $k$ sits at the FM time $t_k = 1 - k/K$, so every tensor that leaves the class is on the course arrow. The
schedule is the one of Ho et al. (2020): $K = 1000$ values $\beta_k$ linear from $10^{-4}$ to $0.02$, and

$$\bar\alpha_k = \prod_{j \le k} (1 - \beta_j),\qquad \alpha_k = \sqrt{\bar\alpha_k},\quad \sigma_k = \sqrt{1-\bar\alpha_k}.$$

The ODE sampler and the weighting check of TODO 1.3 need $\dot\alpha_t$ and $\dot\sigma_t$, which a grid does not
have. The given method `alpha_bar_fm` extends the product to every real $t \in [0,1]$ (with the Gamma function;
its docstring has the formula). It passes through every node of the grid, and `rate` and `coeffs` give its
derivatives. The ancestral chain of Part 3 uses only the grid.

In [ ]:
class VPGrid:
    """The DDPM VP path on the FM arrow (TODO 1.1). All tensors float64, indexed by k = 0..K.

    Native DDPM numbers the chain k = 0 (data) .. K (noisiest). This class translates it once:
    t[k] = 1 - k/K is the FM time, alpha[k] and sigma[k] the coefficients of
    X_{t_k} = alpha_k X_1 + sigma_k X_0. Row k = 0 is the data end (alpha = 1, sigma = 0) and is
    never a training time; rows 1..K are. Training code reads (t, alpha, sigma) and never k's arrow.
    """

    def __init__(self, n_steps: int = K, beta_min: float = BETA_MIN, beta_max: float = BETA_MAX):
        self.n_steps, self.beta_min, self.beta_max = n_steps, beta_min, beta_max
        self.delta = (beta_max - beta_min) / (n_steps - 1)           # step of the beta line
        self.c = (1.0 - beta_min) / self.delta                       # 1 - beta_j = delta (c - j + 1)
        k = torch.arange(n_steps + 1, dtype=torch.float64)
        # TODO 1.1 (a): the five (K + 1,) float64 rows of the grid, indexed by k = 0..K (k is given above).
        #   self.t          the FM time of row k: row 0 is the data end (t = 1), row K the noisiest (t = 0)
        #   self.beta       beta[0] = 0, then the K betas of Ho et al., linear from beta_min to beta_max
        #   self.alpha_bar  the running product of (1 - beta) (torch.cumprod), so alpha_bar[0] = 1
        #   self.alpha, self.sigma   the coefficients of X_t = alpha X_1 + sigma X_0 (VP: alpha^2 + sigma^2 = 1)
        raise NotImplementedError

    def alpha_bar_fm(self, t: torch.Tensor) -> torch.Tensor:
        """alpha_bar at FM time t: the smooth schedule, equal to the grid's product at every node.

        alpha_bar(kappa) = Delta^kappa Gamma(c + 1) / Gamma(c + 1 - kappa), kappa = K (1 - t), written
        with the Stirling series of log Gamma: two lgamma values near 5e4 cancel to 1e-10, this to 1e-13.
        The native position kappa stays inside this method and rate().
        """
        kappa = self.n_steps * (1.0 - t)
        z1, z2 = self.c + 1.0, self.c + 1.0 - kappa
        r = lambda z: 1 / (12 * z) - 1 / (360 * z**3) + 1 / (1260 * z**5)    # noqa: E731
        log_ab = (kappa * torch.log(self.delta * z2) - kappa - (self.c + 0.5) * torch.log1p(-kappa / z1)
                  + r(z1) - r(z2))
        return torch.exp(log_ab)

    def rate(self, t: torch.Tensor) -> torch.Tensor:
        """d/dt log alpha_bar of the smooth schedule, positive on [0, 1]: -K (log Delta + digamma(c + 1 - kappa))."""
        z2 = self.c + 1.0 - self.n_steps * (1.0 - t)
        dlog = torch.log(self.delta * z2) - 1 / (2 * z2) - 1 / (12 * z2**2) + 1 / (120 * z2**4) - 1 / (252 * z2**6)
        return -self.n_steps * dlog

    def coeffs(self, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """(alpha_t, sigma_t, dalpha_t, dsigma_t) of the smooth schedule (TODO 1.1, the interpolation).

        Valid on the interior domain only, where sigma_t > 0: dsigma_t divides by sigma_t.
        """
        # 1e-6: a float32 time (the ODE state's dtype) rounds t_1 = 0.999 up to 0.99900001; sigma is ~0.01 there
        assert float(t.max()) <= self.t[1] + 1e-6, "interior domain: t <= t_1 = 1 - 1/K (sigma_t > 0)"
        ab = self.alpha_bar_fm(t)
        alpha, sigma = ab.sqrt(), (1.0 - ab).sqrt()
        dalpha = 0.5 * self.rate(t) * alpha                          # d/dt sqrt(alpha_bar) = sqrt(alpha_bar) rate / 2
        dsigma = -alpha * dalpha / sigma                             # from alpha^2 + sigma^2 = 1
        return alpha, sigma, dalpha, dsigma

    def at(self, k: torch.Tensor, like: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """(t_k, alpha_k, sigma_k) for (B,) grid rows k, as (B,), (B,1,1,1), (B,1,1,1) in like's dtype (TODO 1.1)."""
        # TODO 1.1 (b): index the three rows with k and cast them to like's dtype and device. t stays (B,);
        # alpha and sigma get the shape (B, 1, ..., 1) with like.ndim axes, so that they broadcast over images.
        raise NotImplementedError

In [ ]:
grid = VPGrid()
print(f"noisiest row k = K:  t = {float(grid.t[K]):.0f}   alpha = {float(grid.alpha[K]):.4g}   "
      f"sigma = {float(grid.sigma[K]):.6f}   SNR = alpha^2 / sigma^2 = {float(grid.alpha[K]**2 / grid.sigma[K]**2):.3g}")
print("alpha at the noisiest row is exactly zero:", bool(grid.alpha[K] == 0))
print(f"last training row k = 1:  t = {float(grid.t[1]):.3f}   sigma = {float(grid.sigma[1]):.4g}")

The noisiest row is **not** pure noise: $\alpha_K \approx 0.0064$, so the forward chain's
marginal at $t=0$ is the law of $\alpha_K X_1 + \sigma_K X_0$. Its mean is $\alpha_K \mathbb E[X_1]$, not $0$: it is
close to $\mathcal N(0, I)$, and not equal to it. Every sampler
below starts from $\mathcal N(0,I)$ anyway. That start is the **standard terminal approximation**, and it is an
approximation: keep the word.

In [ ]:
# Checkpoint 1.1: your grid against closed forms written independently of your code
g = VPGrid()
betas = np.linspace(BETA_MIN, BETA_MAX, K)    # Ho et al. (2020), in NumPy: independent of your code
alpha_K = math.sqrt(float(np.prod(1 - betas)))
assert abs(float(g.t[0]) - 1.0) < 1e-15 and abs(float(g.t[K])) < 1e-15, \
    "t[0] must be the data end (t = 1) and t[K] the noisiest row (t = 0): the FM arrow, not DDPM's"
assert abs(float(g.alpha[K]) - alpha_K) < 1e-12, "alpha[K] is not sqrt(alpha_bar) at t = 0"
assert torch.allclose(g.alpha**2 + g.sigma**2, torch.ones_like(g.alpha), atol=1e-12), \
    "a VP path has alpha^2 + sigma^2 = 1 at every row (sigma is sqrt(1 - alpha_bar), not 1 - alpha)"
assert np.allclose(g.beta[1:].numpy(), betas, rtol=1e-12) and float(g.beta[0]) == 0.0, \
    "beta[1..K] are the K linear betas of Ho et al.; beta[0] = 0"
assert torch.allclose(g.alpha_bar_fm(g.t), g.alpha_bar, rtol=1e-11), \
    "alpha_bar must be the product of (1 - beta_j): the smooth schedule passes through it at every node"
assert g.t.dtype == torch.float64, "keep the grid in float64"
k = torch.tensor([1, 500, K])
x_like = torch.zeros(3, 1, 28, 28)
t_k, a_k, s_k = g.at(k, x_like)
assert t_k.shape == (3,) and a_k.shape == (3, 1, 1, 1) and s_k.dtype == torch.float32, \
    "at() returns (B,), (B,1,1,1), (B,1,1,1) in like's dtype"
print("Checkpoint 1.1 passed.")

### TODO 1.2 — the minibatch and the DDPM-simple loss

The function below is your `sample_batch` of U3.L1 with the same first lines. The time is a **grid row** now:
$k \sim \mathrm{Unif}\{1,\dots,K\}$, the course DDPM convention, and the network sees $t_k$. The loss is

$$\mathcal L_{\text{DDPM-simple}}(\theta) = \mathbb E_{k,\,X_0,\,(X_1, Y),\,\tilde Y}\,
\big\|\varepsilon_\theta(X_{t_k}, t_k, \tilde Y) - X_0\big\|^2,$$

with $k\sim \mathrm{Unif}\{1,\dots,K\}$, $X_0 \sim \mathcal N(0,I)$, $(X_1, Y)$ an image and its label from the training data (images alone follow $q$), and
$\tilde Y$ the label the network sees ($Y$, or the null label with probability `cfg_dropout`, Part 2). This is
the **discrete** objective of Ho et al. (2020): it is this sum over the finite grid, with no weight, that earns
the name DDPM trainer. The weight function is given; its second mode is the subject of TODO 1.3.

In [ ]:
def time_weight(grid: VPGrid, k: torch.Tensor, loss_mode: str) -> torch.Tensor:
    """The (B,) time weight of each loss mode at grid rows k (the third change of the diff).

    ddpm_simple: 1 at every row. cfm_equiv_eps: B_{t_k}^2, the weight that the velocity loss puts
    on the epsilon residual (TODO 1.3 writes b_coeff and checks the identity).
    """
    if loss_mode == "ddpm_simple":
        return torch.ones(k.shape, dtype=torch.float64)
    if loss_mode == "cfm_equiv_eps":
        return b_coeff(grid, grid.t[k]) ** 2
    raise ValueError(f"unknown loss_mode {loss_mode!r}")

In [ ]:
def sample_batch(train_x: torch.Tensor, train_y: torch.Tensor, batch: int, gen: torch.Generator,
                 grid: VPGrid):
    """One minibatch on the VP grid (TODO 1.2). Returns (k, t, x0, x1, y, xt, target), on the CPU.

    The cfm-scratch function of U3.L1 with two lines changed (path) and one (coordinate).
    k ~ Unif{1, ..., K} is the course DDPM convention: one grid row per example, the data row
    k = 0 excluded. t = t_k is the (B,) FM time the network sees; x0 ~ N(0, I) is the noise.
    """
    idx = torch.randint(0, train_x.shape[0], (batch,), generator=gen)
    x1 = train_x[idx]                                                # (B, 1, 28, 28), data
    y = train_y[idx]                                                 # labels: the class pathway
    x0 = torch.randn(x1.shape, generator=gen, dtype=x1.dtype)        # (B, 1, 28, 28), noise
    # TODO 1.2 (a), in this order, every draw from `gen`:
    #   k      ~ Unif{1, ..., K}, shape (batch,), long          (torch.randint; the data row 0 is excluded)
    #   t, a, s = the FM time and the two coefficients of rows k  (grid.at)
    #   xt     = the VP point X_{t_k}
    #   target = what eps_theta regresses on
    raise NotImplementedError
    return k, t, x0, x1, y, xt, target

In [ ]:
def eps_loss(net: nn.Module, grid: VPGrid, k: torch.Tensor, t: torch.Tensor, xt: torch.Tensor,
             target: torch.Tensor, y: torch.Tensor, loss_mode: str) -> torch.Tensor:
    """The two epsilon losses (TODO 1.2 and 1.3): weighted squared norm over pixels, mean over the batch.

    ddpm_simple     E_{k, X_0, (X_1, Y), Y~} || eps_theta(X_{t_k}, t_k, Y~) - X_0 ||^2            (weight 1)
    cfm_equiv_eps   E_{k, X_0, (X_1, Y), Y~} B_{t_k}^2 || eps_theta(X_{t_k}, t_k, Y~) - X_0 ||^2  (induced by CFM)
    with (X_1, Y) an image and its label, and Y~ the label after cfg_dropout (the y argument).
    """
    # TODO 1.2 (b): call the network as net(xt, t, y) (the U-Net contract: image first, then the (B,) time).
    # Weight each example by time_weight(grid, k, loss_mode), cast to xt. The squared norm of the residual
    # is over all pixels of an example (flatten(1), then sum); the mean is over the batch.
    raise NotImplementedError

In [ ]:
# Checkpoint 1.2: your minibatch and your loss, against two networks whose loss is known
g = VPGrid()
x_tr, y_tr = load_mnist("train")
gen = torch.Generator().manual_seed(1)
k, t, x0, x1, y, xt, target = sample_batch(x_tr, y_tr, 64, gen, g)
assert k.dtype == torch.long and int(k.min()) >= 1 and int(k.max()) <= K, "k is a grid row in {1, ..., K}"
k_many = sample_batch(x_tr[:1].expand(20000, -1, -1, -1), y_tr[:1].expand(20000), 20000,
                      torch.Generator().manual_seed(9), g)[0]
assert int(k_many.min()) == 1 and int(k_many.max()) == K, \
    "over 20 000 draws k must reach both 1 and K and never 0: the data row k = 0 is not a training time"
assert torch.equal(t, g.t[k].float()), "t must be the FM time t_k of the rows you drew"
a = g.alpha[k].float().view(-1, 1, 1, 1); s = g.sigma[k].float().view(-1, 1, 1, 1)
assert torch.allclose(xt, a * x1 + s * x0, atol=1e-6), "xt is alpha_k x1 + sigma_k x0"
assert torch.equal(target, x0), "the epsilon coordinate regresses on the noise sample x0"

class ZeroNet(nn.Module):                 # predicts 0: the loss is the mean squared norm of x0
    def forward(self, x, t, y=None):
        return torch.zeros_like(x)

class OracleNet(nn.Module):               # knows the clean images of this batch: predicts x0 exactly
    def forward(self, x, t, y=None):
        a_ = g.alpha_bar_fm(t.double()).sqrt().float().view(-1, 1, 1, 1)
        s_ = (1 - g.alpha_bar_fm(t.double())).sqrt().float().view(-1, 1, 1, 1)
        return (x - a_ * x1) / s_

zero = eps_loss(ZeroNet(), g, k, t, xt, target, y, "ddpm_simple")
expected = float((x0 ** 2).flatten(1).sum(1).mean())
assert abs(float(zero) - expected) / expected < 1e-6, \
    f"with a zero network the loss is E||x0||^2 = {expected:.2f} (sum over pixels, mean over the batch), got {float(zero):.2f}"
assert float(eps_loss(OracleNet(), g, k, t, xt, target, y, "ddpm_simple")) < 1e-6, \
    "a network that predicts x0 exactly must have zero loss"

class SpyNet(nn.Module):                  # records what eps_loss passes to the network
    def forward(self, x, t, y=None):
        self.seen = (x, t, y)
        return torch.zeros_like(x)

spy = SpyNet()
eps_loss(spy, g, k, t, xt, target, y, "ddpm_simple")
x_seen, t_seen, y_seen = spy.seen
assert y_seen is not None and torch.equal(y_seen, y), "call the network as net(xt, t, y): the labels are part of the input"
assert t_seen.shape == (xt.shape[0],) and t_seen.dtype.is_floating_point and float(t_seen.max()) <= 1.0, \
    "the network gets t as a (B,) float tensor in [0, 1]"
assert x_seen is xt, "the network gets the noisy point xt, image first"
print("Checkpoint 1.2 passed.")

### TODO 1.3 — the weighting distinction (mandatory)

The U3.T2 dictionary writes the velocity in the $\varepsilon$ coordinate as
$u_t(x) = \frac{\dot\alpha_t}{\alpha_t}x + B_t\,\bar\varepsilon_t(x)$, with
$\bar\varepsilon_t(x) = \mathbb E[X_0 \mid X_t = x]$ and the noise coefficient $B_t = \dot\sigma_t - \dot\alpha_t\sigma_t/\alpha_t$. Insert it in the velocity CFM loss, with $\varepsilon_\theta$ in $u_t^\theta$ and $X_0$ in $u_t(X_t \mid X_1)$, and the residual is $B_t$
times the $\varepsilon$ residual, so

$$\big\|u_t^\theta(X_t) - u_t(X_t \mid X_1)\big\|^2 = B_t^2\,\big\|\varepsilon_\theta(X_t,t) - X_0\big\|^2 .$$

Write `b_coeff`. The cell after it checks this identity on one frozen batch, with your function on one side
and the velocity residual on the other.

In [ ]:
def b_coeff(grid: VPGrid, t: torch.Tensor) -> torch.Tensor:
    """B_t = dsigma_t - dalpha_t sigma_t / alpha_t, the noise coefficient of U3.T2 (TODO 1.3).

    Interior domain only (coeffs asserts t <= t_1). For any VP schedule B_t = -rate(t) / (2 sigma_t).
    """
    # TODO 1.3: the noise coefficient of the U3.T2 dictionary, from the four outputs of grid.coeffs(t).
    raise NotImplementedError

In [ ]:
# Checkpoint 1.3 (a): your B_t against the closed form of any VP schedule, B_t = -rate(t) / (2 sigma_t),
# with rate(t) = d/dt log alpha_bar(t) (the given method g.rate)
g = VPGrid()
tt = g.t[1:]                              # every training time
closed = -g.rate(tt) / (2 * (1 - g.alpha_bar_fm(tt)).sqrt())
assert torch.allclose(b_coeff(g, tt), closed, rtol=1e-10), \
    "B_t = dsigma - dalpha * sigma / alpha (the U3.T2 dictionary): check the sign and which term is divided"
i_min = int((closed**2).argmin())
print(f"B_t^2 on the grid: {float(closed[-1]**2):.1f} at the noisiest row, minimum {float(closed[i_min]**2):.1f} "
      f"at t = {float(tt[i_min]):.2f}, {float(closed[0]**2):.1f} at the last row t_1")

# Checkpoint 1.3 (b): the identity on a frozen float64 batch, with a fixed network
torch.manual_seed(0)
net = move_head(make_unet({"model": {"base_ch": 16, "ch_mults": [1, 2, 2], "attn_resolutions": [7],
                                     "num_classes": 10}}).double())
gen = torch.Generator().manual_seed(0)
k, t, x0, x1, y, _, target = sample_batch(x_tr.double(), y_tr, 8, gen, g)
alpha, sigma, dalpha, dsigma = g.coeffs(t)
col = lambda v: v.view(-1, 1, 1, 1)
xt = col(alpha) * x1 + col(sigma) * x0     # the smooth (alpha, sigma): the same pair as the conversion
with torch.no_grad():
    eps_hat = net(xt, t, y)
u_hat = col(dalpha / alpha) * xt + col(dsigma - dalpha * sigma / alpha) * eps_hat
u_cond = col(dalpha) * x1 + col(dsigma) * x0          # the conditional velocity of the VP path
velocity_sq = ((u_hat - u_cond) ** 2).flatten(1).sum(1)
weighted_sq = b_coeff(g, t) ** 2 * ((eps_hat - x0) ** 2).flatten(1).sum(1)
rel = float(((velocity_sq - weighted_sq).abs() / weighted_sq).max())
assert rel < 1e-12, f"velocity residual and B_t^2 times the eps residual disagree (relative {rel:.2e})"
w = time_weight(g, k, "cfm_equiv_eps")
assert torch.allclose(w, b_coeff(g, g.t[k]) ** 2), "cfm_equiv_eps weights each example by B_{t_k}^2"
zero_w = eps_loss(ZeroNet(), g, k, t, xt, target, y, "cfm_equiv_eps")
expected_w = float((b_coeff(g, t) ** 2 * (x0 ** 2).flatten(1).sum(1)).mean())
assert abs(float(zero_w) - expected_w) / expected_w < 1e-10, \
    "eps_loss in cfm_equiv_eps mode: with a zero network the loss is E[B_t^2 ||x0||^2]; apply time_weight per example"
print(f"identity holds on the frozen batch: largest relative difference {rel:.1e}")
print("Checkpoint 1.3 passed.")

Same pointwise target does not mean same training. Both modes have the same minimizer at every $t$ (the
conditional mean of $X_0$ given $X_t$ and $\tilde Y$), because a positive time weight does not move a pointwise minimizer. A finite shared
network is not pointwise: the weight decides how errors at different times trade against each other. The cell below compares the
two gradients of the same network on the same batch.

In [ ]:
grads = {}
for mode in ("ddpm_simple", "cfm_equiv_eps"):
    net.zero_grad()
    eps_loss(net, g, k, t, xt, target, y, mode).backward()
    grads[mode] = torch.cat([p.grad.flatten() for p in net.parameters() if p.grad is not None])
cos = float(torch.nn.functional.cosine_similarity(grads["ddpm_simple"], grads["cfm_equiv_eps"], dim=0))
print(f"cosine between the two gradients: {cos:.4f}  (1 would mean the same direction)")

### TODO 1.4 — conversions for sampling

The samplers of Part 3 need the network's $\hat\varepsilon$ in three other coordinates. Each inverse formula
divides by a coefficient, and each asserts the domain where that coefficient is nonzero: $\alpha_t > 0$ holds on
the whole grid, $\sigma_t > 0$ fails at the data end $t = 1$. The ODE sampler stops at the last grid time
$t_1 = 1 - 1/K$, where $\sigma_{t_1} \approx 0.0100$, and never asks for a conversion at $t = 1$.

In [ ]:
def x1_from_eps(alpha: torch.Tensor, sigma: torch.Tensor, x: torch.Tensor, eps_hat: torch.Tensor) -> torch.Tensor:
    """x_1-prediction (x - sigma_t eps) / alpha_t; needs alpha_t > 0, which holds on the whole grid (TODO 1.4)."""
    assert float(alpha.min()) > 0.0, "x1_from_eps divides by alpha_t"
    # TODO 1.4 (a): the data prediction. alpha and sigma are (B,): view them as (B, 1, 1, 1) and cast to x.
    raise NotImplementedError

In [ ]:
def score_from_eps(sigma: torch.Tensor, eps_hat: torch.Tensor) -> torch.Tensor:
    """score -eps / sigma_t; needs sigma_t > 0, so never at the data end t = 1 (TODO 1.4)."""
    assert float(sigma.min()) > 0.0, "score_from_eps divides by sigma_t"
    # TODO 1.4 (b): the score from the noise prediction (sigma is (B,)).
    raise NotImplementedError

In [ ]:
def velocity_from_eps(grid: VPGrid, t: torch.Tensor, x: torch.Tensor, eps_hat: torch.Tensor) -> torch.Tensor:
    """u_t^theta = (dalpha_t / alpha_t) x + B_t eps_hat (TODO 1.4), on the interior domain.

    The U3.T2 dictionary, with the network's eps_hat in place of eps_bar_t(x) = E[X_0 | X_t = x].
    """
    # TODO 1.4 (c): u_t = c_x x + B_t eps with the two coefficients of the dictionary, computed in float64
    # from grid.coeffs(t.double()) and your b_coeff, then viewed as (B, 1, 1, 1) and cast to x.
    raise NotImplementedError

In [ ]:
# Checkpoint 1.4: with eps_hat equal to the true noise, every conversion returns its exact conditional value
g = VPGrid()
gen = torch.Generator().manual_seed(2)
k, t, x0, x1, y, xt, target = sample_batch(x_tr.double(), y_tr, 16, gen, g)
alpha, sigma, dalpha, dsigma = g.coeffs(t)
assert torch.allclose(x1_from_eps(alpha, sigma, xt, x0), x1, atol=1e-10), "x1_from_eps must invert xt = alpha x1 + sigma x0"
assert torch.allclose(score_from_eps(sigma, x0), -x0 / sigma.view(-1, 1, 1, 1)), \
    "the score of N(alpha x1, sigma^2 I) at xt is -(xt - alpha x1) / sigma^2 = -x0 / sigma"
u = velocity_from_eps(g, t, xt, x0)
u_cond = dalpha.view(-1, 1, 1, 1) * x1 + dsigma.view(-1, 1, 1, 1) * x0
assert torch.allclose(u, u_cond, atol=1e-8), "with the true noise the velocity is dalpha x1 + dsigma x0"
for bad in (lambda: score_from_eps(g.sigma[[0]], x0[:1]), lambda: g.coeffs(torch.tensor([1.0], dtype=torch.float64))):
    try:
        bad()
    except AssertionError:
        continue
    raise AssertionError("a conversion at the data end t = 1 (sigma = 0) must stop with an AssertionError")
print("Checkpoint 1.4 passed.")

### The diff

The cell below prints the diff between the `cfm-scratch` functions of U3.L1 and yours, with docstrings and
comments removed. An added line shows a tag (`[path]`, `[coordinate]`, `[weight]`, `[labels]`) when its comment
starts with that word, as in the solutions: put `# path: ...` and so on after your own lines to see them.

In [ ]:
CFM_SCRATCH = 'def sample_batch(train_x: torch.Tensor, batch: int, gen: torch.Generator):\n    """One CFM minibatch on the CondOT path (TODO 1.1). Returns (t, x0, x1, xt, target).\n\n    t ~ Unif[0, 1] has shape (B,): the code-level FM arrow, t = 0 noise and t = 1 data.\n    x1 ~ q is a row of the training set, x0 ~ N(0, I_2), both (B, 2). Every draw comes from `gen`.\n    """\n    idx = torch.randint(0, train_x.shape[0], (batch,), generator=gen)\n    x1 = train_x[idx]                                                # (B, 2), data\n    x0 = torch.randn(x1.shape, generator=gen, dtype=x1.dtype)        # (B, 2), noise\n    t = torch.rand(batch, generator=gen, dtype=x1.dtype)             # (B,)\n    xt = t[:, None] * x1 + (1 - t[:, None]) * x0                     # (B,1) against (B,2)\n    target = x1 - x0                                                 # the conditional velocity\n    return t, x0, x1, xt, target\n\n\ndef cfm_loss(field: nn.Module, t: torch.Tensor, xt: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n    """The minibatch estimate of L_CFM (TODO 1.2): squared norm over coordinates, mean over the batch."""\n    u = field(t, xt)                                                 # u_t^theta(x_t), (B, 2)\n    return ((u - target) ** 2).sum(dim=1).mean()\n'


def code_lines(src: str) -> list[tuple[str, str]]:
    """(code, tag) for each code line of a source string: docstrings, comments and blank lines removed.

    The tag is the first word of a trailing comment when it is one of path, coordinate, weight, labels.
    """
    quotes = ('"' * 3, "'" * 3)
    out, in_doc = [], False
    for line in src.splitlines():
        stripped = line.strip()
        if stripped.startswith(quotes):
            if sum(stripped.count(q) for q in quotes) == 1:
                in_doc = not in_doc
            continue
        if in_doc or not stripped or stripped.startswith("#"):    # full-line comments are not code
            continue
        code_part, _, comment = line.partition("  #")
        tag = re.match(r"\s*(path|coordinate|weight|labels)\b", comment)
        out.append((code_part.rstrip(), tag.group(1) if tag else ""))
    return out


def thesis_diff(old_src: str, new_src: str) -> list[str]:
    """The unified diff of two sources' code lines (no context lines); each added line shows its tag."""
    old, new = code_lines(old_src), code_lines(new_src)
    tags = {c: t for c, t in new}
    lines = []
    for line in difflib.unified_diff([c for c, _ in old], [c for c, _ in new], "cfm-scratch (U3.L1)",
                                     "ddpm_simple (U3.L2)", lineterm="", n=0):
        if line.startswith("+") and not line.startswith("+++") and tags.get(line[1:]):
            line = f"{line}      [{tags[line[1:]]}]"
        lines.append(line)
    return lines



diff = thesis_diff(CFM_SCRATCH, inspect.getsource(sample_batch) + "\n" + inspect.getsource(eps_loss))
print("\n".join(diff))
changed = sum(1 for l in diff if l[:1] in "+-" and l[:3] not in ("+++", "---"))
print(f"\n{changed} changed lines")

**What the diff does not show.** The infrastructure stays: the U-Net instead of the MLP of U3.L1 (an image
model, same field contract `(x, t)` with $t$ a `(B,)` tensor on the FM arrow), AdamW with the warmup-cosine
recipe, the training loop, the seeds. Two lines of the diff are infrastructure too: the signatures, and
the labels `y`, which feed the class pathway of the U-Net. The three tags are the mathematics.

### The gradient-check ritual

The ritual of every lab: autograd against a central difference, in float64, along one fixed direction of the
first layer. Everything random is frozen first: $t$, $X_0$, $X_1$, the labels **and the label-dropout mask**.
A fresh draw between $+h$ and $-h$ would measure the change of the batch, not of the parameters. The head is
moved by a fixed amount first: at exactly zero, every gradient upstream of it is exactly zero, and the check
would compare $0$ with $0$. The last line shows that case.

In [ ]:
def directional_check(net: nn.Module, grid: VPGrid, batch: tuple, loss_mode: str, *, h: float = 1e-6,
                      seed: int = 0) -> dict:
    """Autograd against a central difference, in float64, along one fixed direction of conv_in.weight.

    `batch` is ONE frozen minibatch (k, t, x0, xt, y_in), labels already dropped: nothing is drawn
    between the +h and the -h evaluations. conv_in is the tensor farthest from the head.
    """
    f = copy.deepcopy(net).double()
    k, t, x0, xt, y_in = batch
    t, x0, xt = t.double(), x0.double(), xt.double()
    p = f.conv_in.weight
    v = torch.randn(p.shape, generator=torch.Generator().manual_seed(seed), dtype=torch.float64)
    v = v / v.norm()
    grad = torch.autograd.grad(eps_loss(f, grid, k, t, xt, x0, y_in, loss_mode), p)[0]
    autodiff = float((grad * v).sum())
    with torch.no_grad():
        p.add_(h * v)
        plus = float(eps_loss(f, grid, k, t, xt, x0, y_in, loss_mode))
        p.add_(-2 * h * v)
        minus = float(eps_loss(f, grid, k, t, xt, x0, y_in, loss_mode))
        p.add_(h * v)
    central = (plus - minus) / (2 * h)
    rel = abs(autodiff - central) / max(abs(autodiff), abs(central), 1e-300)
    return {"autodiff": autodiff, "central": central, "rel_err": rel}

In [ ]:
# Checkpoint 1.5: the ritual on both loss modes, then the zero-head control
g = VPGrid()
gen = torch.Generator().manual_seed(0)
k, t, x0, x1, y, xt, target = sample_batch(x_tr.double(), y_tr, 8, gen, g)
y_in = drop_labels(y, 0.5, gen)                     # frozen ONCE, before the +h and -h evaluations
torch.manual_seed(0)
tiny = make_unet({"model": {"base_ch": 16, "ch_mults": [1, 2, 2], "attn_resolutions": [7], "num_classes": 10}}).double()
for mode in ("ddpm_simple", "cfm_equiv_eps"):
    r = directional_check(move_head(tiny), g, (k, t, x0, xt, y_in), mode)
    print(f"{mode:14s} autodiff {r['autodiff']:+.10e}   central {r['central']:+.10e}   rel {r['rel_err']:.1e}")
    assert r["rel_err"] < 1e-6 and abs(r["autodiff"]) > 1e-8, f"{mode}: the ritual fails: {r}"
r0 = directional_check(tiny, g, (k, t, x0, xt, y_in), "ddpm_simple")
print(f"zero head      autodiff {r0['autodiff']:+.1e}   central {r0['central']:+.1e}   (a check of nothing)")
print("Checkpoint 1.5 passed.")

## Part 2 — Train on MNIST

The configuration of the course checkpoint. The class pathway of the U-Net is active (`num_classes = 10`), and
one field is new: `cfg_dropout: 0.1` replaces each label by a null label (row 10 of the label table) with
probability 0.1 during training. Its purpose is the subject of U3.T4 and U3.L3; today it is a field of the config.

In [ ]:
meta = json.loads(CKPT.with_suffix(".json").read_text())
COURSE_CFG = meta["config"]
print(json.dumps(COURSE_CFG, indent=1))

### The live run

A small run of your own code, to see the pipeline move: the same configuration with `base_ch = 16`
(300 steps on a GPU, about 16 s on the reference machine; on a CPU the cell runs
fewer steps and says how many). Its loss curve is the only output; the samples of Part 3 come from the course
checkpoint, trained with the full configuration.

In [ ]:
def warmup_cosine(step: int, total: int, warmup: int) -> float:
    """The recipe-card learning-rate factor: linear warmup, then cosine decay to zero."""
    if step < warmup:
        return step / max(1, warmup)
    return 0.5 * (1.0 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


def train_step(net: nn.Module, opt: torch.optim.Optimizer, grid: VPGrid, train_x: torch.Tensor,
               train_y: torch.Tensor, cfg: dict, gen: torch.Generator) -> float:
    """One optimizer step: the minibatch on the CPU (as in U3.L1), then the step on the network's device."""
    dev = next(net.parameters()).device
    tr = cfg["train"]
    k, t, x0, x1, y, xt, target = sample_batch(train_x, train_y, tr["batch"], gen, grid)
    y_in = drop_labels(y, tr["cfg_dropout"], gen)                    # the cfg_dropout field
    loss = eps_loss(net, grid, k, t.to(dev), xt.to(dev), target.to(dev), y_in.to(dev), tr["loss_mode"])
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    return float(loss.detach())

In [ ]:
live_cfg = copy.deepcopy(COURSE_CFG)
live_cfg["model"]["base_ch"] = 16
live_iters = 300 if ON_GPU else 60
torch.manual_seed(0)
live_net = make_unet(live_cfg).to(DEVICE)
g = VPGrid()
x_tr, y_tr = load_mnist("train")
gen = torch.Generator().manual_seed(1)
opt = torch.optim.AdamW(live_net.parameters(), lr=live_cfg["train"]["lr"], weight_decay=live_cfg["train"]["weight_decay"])
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: warmup_cosine(s, live_iters, min(500, live_iters // 3)))
losses, t0 = [], time.perf_counter()
for step in range(live_iters):
    losses.append(train_step(live_net, opt, g, x_tr, y_tr, live_cfg, gen))
    sched.step()
print(f"{live_iters} steps on {DEVICE} in {time.perf_counter() - t0:.0f} s; last 20 steps: mean loss {np.mean(losses[-20:]):.1f}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses, color="#003366"); ax.set_xlabel("step"); ax.set_ylabel("DDPM-simple loss"); ax.set_yscale("log")
display(fig); plt.close(fig)

### The course checkpoint, and its provenance

The checkpoint trained 20000 steps at `base_ch = 32` with `ddpm_simple` and the seed of its config
(1337 s on the reference GPU). Before you use it, check that what you load is what the record
says: the config hash, the hash of the weights file, and the MLflow export agree.

In [ ]:
def config_hash(config: dict) -> str:
    """Hash of the RESOLVED config (sorted keys), 12 hex digits."""
    blob = json.dumps(config, sort_keys=True, default=str).encode()
    return hashlib.sha256(blob).hexdigest()[:12]


payload = torch.load(CKPT.with_suffix(".pt"), map_location="cpu", weights_only=False)
tags = json.loads((REF / "mlflow_export" / "tags.json").read_text())
weights_sha = hashlib.sha256(CKPT.with_suffix(".pt").read_bytes()).hexdigest()
assert config_hash(payload["config"]) == meta["config_hash"] == tags["config_hash"], "config hash mismatch"
assert weights_sha == meta["weights_sha256"], "the weights file is not the one the record describes"
assert int(tags["seed"]) == meta["seed"] == payload["config"]["seed"], "seed mismatch: MLflow tags, record, checkpoint"
assert payload["config"]["train"]["cfg_dropout"] == 0.1, "the course checkpoint carries cfg_dropout 0.1"
course = make_unet(payload["config"])
course.load_state_dict(payload["state_dict"])
course = course.to(DEVICE).eval()
print(f"config hash {meta['config_hash']}, seed {meta['seed']}, git {meta['git_commit'][:10]}"
      f"{' (dirty)' if meta['git_dirty'] else ''}: verified")

## Part 3 — Two samplers, one model

### TODO 3.1 — the deterministic ODE/PF sampler

Convert $\hat\varepsilon$ to the velocity of TODO 1.4 and integrate it from $t=0$ to $t_1 = 1 - 1/K$ with the
course solver. This is a **continuous** deterministic model, induced by the smooth schedule and the network; it is
not the finite DDPM chain. Its cost is the solver's NFE: the number of right-hand-side calls, each one network call.
`odeint` wants a flat state `(B, d)`, so the function flattens the images and the right-hand side views them back.

In [ ]:
@torch.no_grad()
def sample_ode(net: nn.Module, grid: VPGrid, x0: torch.Tensor, y: torch.Tensor, *,
               method: str = "adaptive_heun", n_steps: int | None = None,
               rtol: float | None = None, atol: float | None = None):
    """Deterministic ODE/PF sampler (TODO 3.1): integrate u from t = 0 to t_1 = 1 - 1/K.

    The field is the continuous model induced by the smooth schedule and the network, not the
    finite DDPM chain. It stops at the last grid time t_1, where sigma_t > 0, and returns x_{t_1}
    (no division at the data end). Returns (SolverOutput, images): read .nfe from the first, before
    any arithmetic. Starting from N(0, I) at t = 0 is the terminal approximation: the forward
    marginal there is the law of alpha_K X_1 + sigma_K X_0, whose mean alpha_K E[X_1] is small, not zero.
    """
    shape = x0.shape                                                 # (B, 1, 28, 28); odeint wants (B, d)

    def rhs(t, x):                                                   # f(t, x), time first
        # TODO 3.1: odeint passes a scalar t and the flat state x of shape (B, 784). Expand t to (B,),
        # view x as images, call the network, convert eps_hat to the velocity, and return it flat.
        raise NotImplementedError
    sol = odesolve.odeint(rhs, x0.flatten(1), 0.0, float(grid.t[1]), method=method, n_steps=n_steps,
                          rtol=rtol, atol=atol)
    return sol, sol.as_subclass(torch.Tensor).view(shape)

### TODO 3.2 — the DDPM ancestral update (given as a formula)

The ancestral chain moves down the grid, from row $j$ to a less noisy row $i < j$. Write
$a_{j|i} = \alpha_j/\alpha_i$ and $\rho_{j|i} = \sigma_j^2 - a_{j|i}^2\sigma_i^2$, the scale and the variance of the
**forward** move from row $i$ to row $j$. With $\hat x_1$ the data prediction at row $j$, the update is

$$\begin{aligned}
x_{t_i} &\sim \mathcal N\big(m_{i|j},\ \tilde\rho_{i|j}\, I\big), \qquad
\tilde\rho_{i|j} = \frac{\sigma_i^2\, \rho_{j|i}}{\sigma_j^2}, \\
m_{i|j} &= \frac{\alpha_i\, \rho_{j|i}}{\sigma_j^2}\,\hat x_1 + \frac{a_{j|i}\,\sigma_i^2}{\sigma_j^2}\,x_{t_j} .
\end{aligned}$$

Finite-step formula from Gaussian conditioning; U3.T3 connects its continuum limit to reverse-time SDEs. For
$i = j - 1$ it is the update of Ho et al. (2020). The lab uses it and measures it; it does not derive it.
The `z = None` branch is the deterministic DDIM update (Song et al., 2021), given.

In [ ]:
def posterior_step(grid: VPGrid, x: torch.Tensor, eps_hat: torch.Tensor, j: int, i: int,
                   z: torch.Tensor | None) -> torch.Tensor:
    """One move from grid row j to the less noisy row i < j (TODO 3.2 and the DDIM cell of 3.3).

    With z given: the DDPM ancestral update, a draw from the Gaussian posterior
    q(x_{t_i} | x_{t_j}, X_1 = x1_hat) of the forward chain, with x1_hat from eps_hat. For i = j - 1
    this is the update of Ho et al. 2020. With z = None: the DDIM update (eta = 0) of Song et al.
    2021, x_{t_i} = alpha_i x1_hat + sigma_i eps_hat.
    """
    a_j, s_j, a_i, s_i = (grid.alpha[j], grid.sigma[j], grid.alpha[i], grid.sigma[i])
    x1_hat = x1_from_eps(a_j.view(1), s_j.view(1), x, eps_hat)
    if z is None:
        return a_i.to(x) * x1_hat + s_i.to(x) * eps_hat
    # TODO 3.2: the ancestral update. Use the box above the cell: the forward move from row i to row j
    # scales by a_ji = alpha_j / alpha_i and adds variance var_ji; the posterior mean is a combination of
    # x1_hat and x; return mean + sqrt(posterior variance) * z. Every coefficient is float64: cast with .to(x).
    raise NotImplementedError

In [ ]:
def substeps(grid: VPGrid, n_steps: int) -> list[int]:
    """n_steps + 1 grid rows from K (noisiest) down to 0 (data), evenly spaced in k."""
    rows = np.linspace(grid.n_steps, 0, n_steps + 1).round().astype(int)
    assert len(set(rows)) == n_steps + 1, "n_steps must be at most K"
    return [int(r) for r in rows]


@torch.no_grad()
def sample_chain(net: nn.Module, grid: VPGrid, x0: torch.Tensor, y: torch.Tensor, n_steps: int, *,
                 stochastic: bool, gen: torch.Generator | None = None) -> dict:
    """The ancestral chain (stochastic=True) or DDIM (False) on n_steps grid moves from row K to 0.

    Starts from x0 ~ N(0, I) at row K: the terminal approximation (alpha_K is small, not zero).
    Returns the samples and the two cost counters of a chain: sampler_steps and model_forwards.
    """
    rows = substeps(grid, n_steps)
    x = x0
    forwards = 0
    for j, i in zip(rows[:-1], rows[1:]):
        t = grid.t[j].to(x).expand(x.shape[0])
        eps_hat = net(x, t, y)
        forwards += 1
        noisy = stochastic and i > 0                                 # the last move (i = 0) adds no noise
        z = torch.randn(x.shape, generator=gen).to(x) if noisy else None
        x = posterior_step(grid, x, eps_hat, j, i, z)
    return {"x": x, "sampler_steps": n_steps, "model_forwards": forwards}

In [ ]:
# Checkpoint 3.1 and 3.2: Gaussian data, where the exact eps is known in closed form
g = VPGrid()
m, sd = 0.3, 0.5                                  # data X_1 ~ N(m, sd^2 I) on 784 pixels

class GaussEps(nn.Module):                        # the exact E[X_0 | X_t = x] for that data
    def forward(self, x, t, y=None):
        assert t.shape == (x.shape[0],), "the network gets t as a (B,) tensor: expand the solver's scalar t"
        ab = g.alpha_bar_fm(t.double()).to(x).view(-1, 1, 1, 1)
        a_, s_ = ab.sqrt(), (1 - ab).sqrt()
        return s_ * (x - a_ * m) / (a_**2 * sd**2 + s_**2)

x0 = torch.randn(400, 1, 28, 28, generator=torch.Generator().manual_seed(4), dtype=torch.float64)
y0 = torch.zeros(400, dtype=torch.long)
sol, x_ode = sample_ode(GaussEps(), g, x0, y0, method="rk4", n_steps=200)
assert isinstance(sol, odesolve.SolverOutput) and sol.nfe == 800, "return odeint's SolverOutput unchanged (rk4: 4 calls per step)"
print(f"ODE:       mean {float(x_ode.mean()):.4f} (target {m})   std {float(x_ode.std()):.4f} (target {sd})   NFE {sol.nfe}")
assert abs(float(x_ode.mean()) - m) < 0.01 and abs(float(x_ode.std()) - sd) < 0.01, "the ODE does not carry N(0, I) to the data"

j, i = 400, 399                                   # one ancestral step against Ho et al.'s noise-form update
xj = torch.randn(400, 1, 28, 28, generator=torch.Generator().manual_seed(5), dtype=torch.float64)
eps = GaussEps()(xj, g.t[j].expand(400))
z = torch.randn(xj.shape, generator=torch.Generator().manual_seed(6), dtype=torch.float64)
beta = g.beta[j]
ho_mean = (xj - beta / g.sigma[j] * eps) / (1 - beta).sqrt()
ho_var = (1 - g.alpha_bar[i]) / (1 - g.alpha_bar[j]) * beta
assert torch.allclose(posterior_step(g, xj, eps, j, i, z), ho_mean + ho_var.sqrt() * z, atol=1e-10), \
    "one ancestral step must equal (x - beta_k / sigma_k eps) / sqrt(1 - beta_k) + sqrt(beta_tilde_k) z"

res = sample_chain(GaussEps(), g, x0, y0, 1000, stochastic=True, gen=torch.Generator().manual_seed(7))
print(f"ancestral: mean {float(res['x'].mean()):.4f}   std {float(res['x'].std()):.4f}   "
      f"steps {res['sampler_steps']}   model forwards {res['model_forwards']}")
assert abs(float(res["x"].mean()) - m) < 0.01 and abs(float(res["x"].std()) - sd) < 0.01, \
    "the ancestral chain does not reach the data distribution: check the posterior variance"
print("Checkpoints 3.1 and 3.2 passed.")

### TODO 3.3 — one model, three samplers, one noise bank

Use `harness.panel` with the **same** initial-noise bank and the same labels (four rows of the digits 0 to 9) for
three samplers of the course checkpoint: the ODE (`adaptive_heun`, `rtol = atol = 1e-3`), the ancestral chain on all
$K$ rows, and DDIM on 50 rows. Print the cost of each run with the counter that belongs to it: the ODE reports
solver NFE; the two chains report `sampler_steps` and `model_forwards`. A chain step is not an NFE: if you compare
costs, compare model forwards (or wall-clock), and name the quantity.

In [ ]:
bank = torch.randn(40, 1, 28, 28, generator=torch.Generator().manual_seed(2026)).to(DEVICE)
labels = (torch.arange(40) % 10).to(DEVICE)
g = VPGrid()
to_unit = lambda x: ((x + 1) / 2).clamp(0, 1)
COST = {}
# TODO 3.3: three functions eps_bank -> images in [0, 1], each passed to display(harness.panel(fn, bank,
# nrow=10, title=...)) (panel switches matplotlib to Agg, so plt.show would show nothing). Inside each, store the cost in COST["ode"], COST["ancestral"], COST["ddim"]:
#   ODE: the SolverOutput's .nfe; chains: sampler_steps and model_forwards. Seed the ancestral noise with 5.
raise NotImplementedError

In [ ]:
# Checkpoint 3.3: three panels, the right counter for each
assert set(COST) == {"ode", "ancestral", "ddim"}, "one cost entry per sampler"
assert set(COST["ode"]) == {"nfe"} and COST["ode"]["nfe"] > 0, "the ODE reports solver NFE, and only that"
assert COST["ancestral"] == {"sampler_steps": K, "model_forwards": K}, "the ancestral chain: K steps, K model forwards"
assert COST["ddim"] == {"sampler_steps": 50, "model_forwards": 50}, "DDIM on 50 rows: 50 steps, 50 model forwards"
print("Checkpoint 3.3 passed.")

**Observation (TODO 3.3).** Same noise bank, same labels, same network. Describe what you see, sampler against
sampler: which panels look alike digit by digit, which do not, and what that suggests about which samplers are
deterministic functions of the initial noise. Phenomenology only, two to four sentences.

## Part 4 — Azula cross-check: the dictionary, numerically

Azula speaks the diffusion dialect: native time $s \in [0,1]$ runs clean → noisy, a schedule returns
$(\alpha_s, \sigma_s)$, and a denoiser returns a posterior whose mean estimates the clean data, $\mathbb E[X \mid x_s]$. The course speaks
the FM dialect. Two adapters translate: the schedule answers Azula at FM time $t = 1-s$ (the Rosetta), and the
denoiser converts the course $\hat\varepsilon$ to $\hat x_1$. Native time exists only inside the two adapters and in the checkpoint's calls to Azula.

Azula's own `VPSchedule` is a different VP family ($\log\alpha_s = s^2 \log\alpha_{\min}$, with a floor on
$\sigma$). It does not agree with the course schedule at any setting, so the cross-check does not compare those
two; the cell after the adapters prints them side by side for reference. The cross-check compares Azula's
**DDPM and DDIM steps** with yours, under the same noise. Azula's `denoiser.loss` has its own weighting; it is
not compared with any course loss.

In [ ]:
class CourseSchedule(azula.noise.Schedule):
    """The course VP schedule in Azula's dialect (TODO 4.1): native time s = 1 - t, clean -> noisy.

    Azula asks for (alpha_s, sigma_s) at native time s. The adapter answers with the course
    coefficients at FM time t = 1 - s. This is the only place where native time exists.
    """

    def __init__(self, grid: VPGrid):
        self.grid = grid

    def __call__(self, s: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO 4.1: Azula passes its native time s. Translate it to the FM time, read alpha_bar there from
        # the smooth schedule (in float64), and return (alpha, sigma) in s's dtype.
        raise NotImplementedError

In [ ]:
class EpsDenoiser(azula.denoise.Denoiser):
    """The course epsilon network as an Azula denoiser (TODO 4.2): Azula expects E[X | x_s], the clean image."""

    def __init__(self, net: nn.Module, grid: VPGrid):
        super().__init__()
        self.net, self.schedule = net, CourseSchedule(grid)

    def forward(self, x_s: torch.Tensor, s: torch.Tensor, y: torch.Tensor | None = None, **kwargs):
        s = s.expand(x_s.shape[0]) if s.ndim == 0 else s
        # TODO 4.2: the network wants FM time; Azula gave native time s. Call the course network, convert its
        # eps_hat to the clean-image prediction Azula expects, and wrap it as azula.denoise.DiracPosterior(mean=...).
        raise NotImplementedError

In [ ]:
# Checkpoint 4: your steps against Azula's, same noise (float64, a small fixed network)
g = VPGrid()
torch.manual_seed(0)
net = move_head(make_unet({"model": {"base_ch": 16, "ch_mults": [1, 2, 2], "attn_resolutions": [7],
                                     "num_classes": 10}}).double())
den = EpsDenoiser(net, g)
s_nodes = 1.0 - g.t                                  # native time of each grid row
a_az, s_az = CourseSchedule(g)(s_nodes)
assert torch.allclose(a_az, g.alpha, atol=1e-12) and torch.allclose(s_az, g.sigma, atol=1e-12), \
    "the adapter must give Azula the course coefficients at native time s = 1 - t"
x = torch.randn(8, 1, 28, 28, generator=torch.Generator().manual_seed(3), dtype=torch.float64)
y8 = torch.arange(8) % 10
worst = {"ddpm": 0.0, "ddim": 0.0}
for j, i in ((K, K - 1), (500, 499), (2, 1), (1, 0), (K, 980), (600, 400), (50, 0)):
    eps_hat = net(x, g.t[j].expand(8), y8)
    torch.manual_seed(7); z = torch.randn_like(x)
    ours = posterior_step(g, x, eps_hat, j, i, z if i > 0 else None)
    torch.manual_seed(7); theirs = azula.sample.DDPMSampler(den).step(x, s_nodes[j], s_nodes[i], y=y8)
    worst["ddpm"] = max(worst["ddpm"], float((ours - theirs).abs().max()))
    ours = posterior_step(g, x, eps_hat, j, i, None)
    theirs = azula.sample.DDIMSampler(den, eta=0.0).step(x, s_nodes[j], s_nodes[i], y=y8)
    worst["ddim"] = max(worst["ddim"], float((ours - theirs).abs().max()))
chain = (sample_chain(net, g, x, y8, 50, stochastic=False)["x"]
         - azula.sample.DDIMSampler(den, eta=0.0, steps=50, silent=True, dtype=torch.float64)(x, y=y8)).abs().max()
print(f"largest difference, one step:  DDPM {worst['ddpm']:.1e}   DDIM {worst['ddim']:.1e}")
print(f"largest difference, a 50-step DDIM chain: {float(chain):.1e}")
assert max(worst.values()) < 1e-9 and float(chain) < 1e-9, "your steps and Azula's disagree: read the two adapters and your posterior_step (TODO 3.2) first"
print("Checkpoint 4 passed.")

print("\n   t     course alpha  course sigma   Azula VPSchedule alpha  sigma  (s = 1 - t, defaults)")
az = azula.noise.VPSchedule()
for t_ in (0.0, 0.25, 0.5, 0.75, 0.9):
    tt = torch.tensor([t_], dtype=torch.float64)
    a_c, s_c = CourseSchedule(g)(1 - tt)
    a_a, s_a = az(1 - tt.float())
    print(f"  {t_:.2f}   {float(a_c):.4f}        {float(s_c):.4f}         {float(a_a):.4f}                  {float(s_a):.4f}")

**Source read (Part 4).** Open `azula/sample.py` and `azula/denoise.py` (`python -c "import azula, os;
print(os.path.dirname(azula.__file__))"` prints the folder) and fill the table.

| | Azula | course |
|---|---|---|
| time variable and its direction | | |
| what the schedule returns | | |
| what the denoiser predicts | | |
| the conversion your adapter applies | | |

## Part 5 — FID, and a first look at NFE against FID

### TODO 5.1 — FID of the ODE sampler

The harness fixes the protocol: the pinned MNIST extractor, its preprocessing, and the course sample count
`harness.MNIST_FID_SAMPLES` = 10000. Score that many ODE samples of the course checkpoint (balanced labels,
1 000 per digit) against the MNIST **test** split. On a CPU, 10 000 ODE samples take too long for a session: the
cell reads the reference value instead and says so.

In [ ]:
def test_ref():
    x, _ = load_mnist("test")
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset((x + 1) / 2), batch_size=1024)
    return harness.ref_stats(loader, EXTRACTOR, dataset="mnist-u3l2-test", split="test",
                             cache_dir=DATA_DIR / "refstats")

@torch.no_grad()
def ode_samples(net, n, *, method, n_steps=None, rtol=None, atol=None, seed=12, batch=1000):
    """n samples, n / 10 per digit, in [0, 1]; returns them and the NFE of each solve."""
    gen, out, nfes = torch.Generator().manual_seed(seed), [], []
    labels = torch.arange(n) % 10
    for i in range(0, n, batch):
        yb = labels[i:i + batch].to(DEVICE)
        x0 = torch.randn(len(yb), 1, 28, 28, generator=gen).to(DEVICE)
        sol, x = sample_ode(net, VPGrid(), x0, yb, method=method, n_steps=n_steps, rtol=rtol, atol=atol)
        nfes.append(sol.nfe)
        out.append(((x + 1) / 2).clamp(0, 1).cpu())
    return torch.cat(out), nfes

EXTRACTOR = harness.MNISTFeatures(device=DEVICE)
REF_FID = json.loads((REF / "fid_table.json").read_text())

In [ ]:
# TODO 5.1: on a GPU, draw harness.MNIST_FID_SAMPLES samples with ode_samples(course, ..., method="adaptive_heun",
# rtol=1e-3, atol=1e-3), score them into fid_ode = harness.fid(samples, test_ref(), EXTRACTOR) (keep the returned
# FIDValue: it carries its protocol), and print it with the median NFE. On a CPU, set fid_ode = None and print
# REF_FID["rows"]["ode-adaptive_heun"], saying that it is the reference value.
fid_ode = None
raise NotImplementedError

### TODO 5.2 — FID against NFE, a first look

The fixed-step Euler solver at NFE $\in \{5, 10, 20, 50\}$: one network call per step, so NFE is the step count.
Plot FID against NFE on a log axis with the plot conventions of `common/payoff.py`. This frontier is the object
U3.T4/U5 attack; today it is just measured.

In [ ]:
NFE_SWEEP = (5, 10, 20, 50)
# TODO 5.2: one FID per NFE in NFE_SWEEP (method="euler", n_steps=nfe) on a GPU, kept as the FIDValue that
# harness.fid returns, else the float REF_FID["rows"][f"ode-euler-{nfe}"]["fid"]. Store them in fids, plot fids against NFE_SWEEP (log x),
# style the axes with payoff._style(ax), and show the figure with display(fig).
fids = []
raise NotImplementedError

In [ ]:
# Checkpoint 5: the protocol, the trend, and (on a GPU) the reference values
assert len(fids) == 4 and all(math.isfinite(f) and f > 0 for f in fids), "one positive FID per NFE value"
assert all(a > b for a, b in zip(fids, fids[1:])), \
    "FID must fall as the Euler NFE grows on this checkpoint (reference 238.6, 13.7, 6.2, 4.4): one sample set per NFE"
if ON_GPU:
    for f in fids + [fid_ode]:
        assert isinstance(f, harness.FIDValue) and f.n_samples == harness.MNIST_FID_SAMPLES, \
            "score MNIST_FID_SAMPLES samples with harness.fid and keep its FIDValue: a smaller n is another protocol"
    ref_vals = [REF_FID["rows"][f"ode-euler-{n}"]["fid"] for n in NFE_SWEEP] + [REF_FID["rows"]["ode-adaptive_heun"]["fid"]]
    for f, r in zip(fids + [fid_ode], ref_vals):
        assert abs(float(f) - r) / r < 0.15, f"FID {float(f):.2f} is more than 15% from the reference {r:.2f}"
print("Checkpoint 5 passed.")
print("\nThe chains, for reference (same n, same extractor; cost counted in model forwards):")
for key, row in REF_FID["rows"].items():
    if row["sampler"] != "ode":
        print(f"  {key:16s} model forwards {row['model_forwards']:5d}   FID {row['fid']:.2f}")

## Close

You kept one network and changed three things: the path, the prediction coordinate, the weight. The same
checkpoint then sampled through a continuous deterministic model and through a finite stochastic chain.

U3.T3 separates the finite DDPM posterior derivation from the reverse-SDE limit, derives the PF-ODE sign-safely,
and shows how to compute the likelihood of the deterministic ODE model induced by this checkpoint.

**Keep the diff.** Save the output of the diff cell with your notebook: the course comes back to it at the end.
The deadlines of PS3 are in the course calendar.